# RAG 文档问答 · 跟教程手敲版

这是跟着教程一步步敲下来的原始记录。**代码按教程原文抄录，逻辑与写法一律未改**——`retreive` 的拼写、`print(f"[{i} {chunks}\n]")` 里打印整个 `chunks`、`EphemeralClient()` 的用法，都保持原样。

> 唯一处理：原稿复制时部分 `print` 的缩进丢失，此处按 Python 语法补回 4 个空格，除此之外逐字未动。

工程化重构后的版本见 [pipeline.py](./pipeline.py)。

## 0. 准备环境

```bash
mkdir rag
cd rag
uv init .
tree .

uv add sentence_transformers chromadb openai python-dotenv

notepad doc.md
# 把要检索的文章粘进去

uv run --with jupyter jupyter lab
```

## 1. 分片

In [ ]:
from typing import List

def split_into_chunks(doc_file:str) -> List[str]:
    with open(doc_file,'r',encoding='utf-8') as file:
        content = file.read()
    return [chunk for chunk in content.split("\n\n")]

chunks = split_into_chunks("doc.md")

for i,chunk in enumerate(chunks):
    print(f"[{i} {chunks}\n]")

## 2. 索引（拿到所有向量值）

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("BAAI/bge-small-zh-v1.5")

def embed_chunk(chunk: str) -> list[float]:
    return embedding_model.encode(chunk).tolist()

test_embedding = embed_chunk("测试内容")
print(len(test_embedding))
print(test_embedding)

### 第一个片段的向量值

In [ ]:
embeddings = [embed_chunk(chunk) for chunk in chunks]
print(len(embeddings))
print(embeddings[0])

## 3. 存入数据库

In [ ]:
import chromadb
chromadb_client = chromadb.EphemeralClient()  # 内存性，脚本运行结束自动清除数据
chromadb_collection = chromadb_client.get_or_create_collection(name="default")

def save_embeddings(chunks: List[str], embeddings: List[List[float]]) -> None:
    ids = [str(i) for i in range(len(chunks))]
    chromadb_collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=ids
    )

save_embeddings(chunks, embeddings)

## 4. 召回

In [ ]:
def retreive(query: str, top_k: int) -> List[str]:
    query_embedding = embed_chunk(query)
    results = chromadb_collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results['documents'][0]

query = "安子卖什么"
retrieved_chunks = retreive(query, 5)

for i, chunk in enumerate(retrieved_chunks):
    print(f"[{i}] {chunk}\n")

## 5. 重排

In [ ]:
from sentence_transformers import CrossEncoder

def rerank(query: str, retrieved_chunks: List[str], top_k: int) -> List[str]:
    cross_encoder = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')
    pairs = [(query, chunk) for chunk in retrieved_chunks]
    scores = cross_encoder.predict(pairs)

    chunk_with_score_list = [(chunk, score)
                             for chunk, score in zip(retrieved_chunks, scores)]
    chunk_with_score_list.sort(key=lambda pair: pair[1], reverse=True)

    return [chunk for chunk, _ in chunk_with_score_list][:top_k]

reranked_chunks = rerank(query, retrieved_chunks, 3)

for i, chunk in enumerate(reranked_chunks):
    print(f"[{i}] {chunk}\n")

## 6. 生成

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

PROMPT_TEMPLATE = """你是一位知识助手，请根据用户的问题和下列片段生成准确的回答。

用户问题：{query}

相关片段：
{context}

请基于上述内容作答，不要编造信息。如果片段中找不到答案，直接回答「文档中没有相关信息」。"""

llm_client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url="https://open.bigmodel.cn/api/paas/v4",
)

def generate(query: str, chunks: list[str]) -> str:
    prompt = PROMPT_TEMPLATE.format(query=query, context="\n\n".join(chunks))
    response = llm_client.chat.completions.create(
        model="glm-4.5-air",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        extra_body={"thinking": {"type": "disabled"}},
    )
    return response.choices[0].message.content.strip()


answer = generate(query, reranked_chunks)
print(answer)

## 7. 配置文件 `.env`

```ini
LLM_API_KEY=""
LLM_BASE_URL="https://open.bigmodel.cn/api/paas/v4"
LLM_MODEL="glm-4.5-air"

LLM_THINKING="disabled"
```

---

## 运行提示（不属于教程原文）

照抄的写法里有几处会报错，留着是为了还原当时的记录，不是 bug 没发现：

| 位置 | 现象 | 原因 |
| --- | --- | --- |
| 1. 分片 | 每行都打印整份 `chunks` | `print(f"[{i} {chunks}\n]")` 里是 `chunks` 而不是 `chunk` |
| 3. 存入数据库 | 数据只在内存，重启即丢 | `EphemeralClient()` 不落盘，工程版换成了 `PersistentClient` |
| 5. 重排 | 看不出分数正负代表什么 | 教程只取排序，没有把 `score` 打出来 |
| 6. 生成 | 无关问题也会被硬答 | 没有阈值，检索到什么都喂给模型 |

这些问题在 [pipeline.py](./pipeline.py) 里都做了处理。